# AML Data Preprocessing

## BASICS

### --- IMPORT LIBRARIES ---

In [ ]:
import torch
import time
import random
import datetime
import numpy as np
import pandas as pd
import networkx as nx
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
from pandas import Timestamp
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader
from torch_geometric.utils import negative_sampling, train_test_split_edges


In [ ]:
# importing functions from functions.py
from functions import *

### --- LOAD DATASET ---

In [ ]:
filename = "/Users/owhy/Documents/Datasets/HI-Small_Trans_balanced.csv"
data = pd.read_csv(filename)
print(f"Shape of DataFrame - {data.shape}")
data.head()

In [ ]:
print("---- info ----")
data.info()

In [ ]:
print("---- basic calculations ----")
data.describe()

#### Null elements?

In [ ]:
print(data.isnull().sum())

#### Fraudulent or Not? - Y labels

In [ ]:
print(f"Number of fraudulent transactions - {len(data[data["Is Laundering"]==1])}")
print(f"Number of non-fraudulent transactions - {len(data[data["Is Laundering"]==0])}")
data[data["Is Laundering"]==1]

In [ ]:
laundering_accounts = list(data[data["Is Laundering"]==1]["Account"])
print(laundering_accounts)
print(f"Laundering Accounts - {laundering_accounts}")
data[data["Account"]==laundering_accounts[0]]

In [ ]:
labels = data["Is Laundering"].to_numpy()
labels

#### Checking similarities between columns

In [ ]:
print("Are Amount Paid entirely equal to Amount Received?\n - " + str(data["Amount Paid"].equals(data["Amount Received"])))
print("Are Currency Received entirely equal to Currency Paid?\n - " + str(data["Payment Currency"].equals(data["Receiving Currency"])))

#### Checking amount of unique categories for Payments

In [ ]:
print(sorted(data["Receiving Currency"].unique()))
print(sorted(data["Payment Currency"].unique()))
print(sorted(data["Payment Format"].unique()))
merged_unique_accounts = pd.concat([data["Account"], data["Account.1"]]).unique()
print(len(merged_unique_accounts))

## NODE MATRIX

In [ ]:
unique_accounts = get_nodes(data)
unique_accounts

#### --- One-hot encoding: currency ---

In [ ]:
node_features = one_hot_encoding(unique_accounts, column="Currency")
node_features

#### --- Normalization and Vectorization ---

In [ ]:
# bank and accounts as independent dataframes for vectorizaiton and normalization purposes
from_bank_col = node_features.pop('Bank')
account_col = node_features.pop('Accounts')

node_labels = pd.DataFrame(account_col)

In [ ]:
# DONE add vectors as individual columns in new dataframe
df = pd.DataFrame(account_col, columns=['Accounts'])
df.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series
vectors = hashing_vectorization(df['Accounts'], vector_size=9)

# Convert vectors into DataFrame
vectors_df = pd.DataFrame(vectors, columns=[f'col_{i}' for i in range(len(vectors[0]))])
result_df = pd.concat([df, vectors_df], axis=1)
accounts_df = result_df.drop(columns=["Accounts"])

In [ ]:
from_bank_binary = [bin(x).split("b")[1] for x in from_bank_col]
binary_lists = [[int(bit) for bit in binary] for binary in from_bank_binary]

In [ ]:
longest_str, len_longest_str = get_longest_string_in_list(from_bank_binary)
binary_lists = make_binary_fixed_length(binary_lists, longest_str)
bin_vectors_df = pd.DataFrame(binary_lists, columns=[f'bin_{i}' for i in range(len(binary_lists[0]))])
# bin_vectors_df

In [ ]:
# DONE normalize vector values to avoid big numbers
from_bank_df = pd.DataFrame(from_bank_col)
accounts_df = pd.DataFrame(accounts_df)

# DONE do not normalize at this point --> create BINARY representation
accounts_df_norm = normalize(accounts_df,0,1)
# accounts_df_norm

#### --- Final Node Features ---

In [ ]:
node_features.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series
accounts_df_norm.reset_index(drop=True, inplace=True) # Ensure the DataFrame has the same number of rows as the original series

node_features = pd.concat([node_features, accounts_df_norm], axis=1)
node_features = pd.concat([node_features, bin_vectors_df], axis=1)
node_features

#### --- Account: Unique Identifier ---

In [ ]:
# DONE add unique random identified
unique_ids_set = set()
while len(unique_ids_set) < len(node_features): # uniqueness kept
    unique_ids_set.add(random.random())
unique_ids = list(unique_ids_set)
node_features.insert(0, "Unique ID", unique_ids)
node_features

#### --- X = Node Feature Matrix ---

In [ ]:
# DONE nodes should be bank accounts and not transactions. Bank accounts have unique receiving currencies and "bank BINs"
x = node_features.to_numpy()
x.shape # [num_nodes x num_features]

## EDGE MATRIX

### --- Finding Links: Mapping ---

In [ ]:
links = [{'source': source, 'destination': destination} for source, destination in zip(data['Account'], data['Account.1'])]

In [ ]:
links[-1]

### --- Edge Features ---

In [ ]:
# DONE add edge features --> create matrix like those for nodes 
edges_df = data[["Timestamp", "Amount Paid", "Payment Currency", "Payment Format"]]
edges_df

#### --- Payment Encoding ---

In [ ]:
edges_amount = edges_df["Amount Paid"].astype(str)
edges_amount = list(edges_amount)

In [ ]:
maximum = str(max(edges_df["Amount Paid"]))
max_len = len(maximum.split(".")[0])
minimum = min(edges_df["Amount Paid"])
minimum = format(minimum, 'f')
min_len = len(str(minimum.split('.')[1]))

new_min, count = count_unused_decimals(minimum)
min_len = min_len - count

number_columns = max_len + min_len
number_columns

In [ ]:
a = split_into_vectors(edges_amount) # INEFFICIENT !!!! # INEFFICIENT !!!!# INEFFICIENT !!!!# INEFFICIENT !!!!# INEFFICIENT !!!!

new_payment_list = encode_payment_amount(a, max_len, min_len) # INEFFICIENT !!!! # INEFFICIENT !!!!# INEFFICIENT !!!!# INEFFICIENT !!!!# INEFFICIENT !!!!
new_payment_list = nested_list_int = [[int(item) for item in sublist] for sublist in new_payment_list]

In [ ]:
# Convert vectors into DataFrame
payment_vectors_df = pd.DataFrame(new_payment_list, columns=[f'payment_{i}' for i in range(len(new_payment_list[0]))])

# ...
edges_features = pd.concat([edges_df, payment_vectors_df], axis=1)
edges_features.drop("Amount Paid", axis='columns')

#### --- One-hot encoding: currency ---

In [ ]:
# DONE convert Currency into one-hot encoding
positions = edges_features["Payment Currency"].str.split(",", expand=True) # creating new columns by splitting receiving currency --> all are added
edges_features["first_position"] = positions[0] # first currency in each row is extracted --> actual currency used and that we want as TRUE
# One-hot encoding
edges_features = pd.concat([edges_features, pd.get_dummies(edges_features["first_position"],dtype='int')], axis=1, join='inner') # effectively adds actual currency to dummy variables/columns
edges_features.drop(["Amount Paid","Payment Currency", "first_position"], axis=1, inplace=True) # drop the axiliary columns
edges_features.head()

In [ ]:
# DONE convert Payment Format
positions_2 = edges_features["Payment Format"].str.split(",", expand=True)
edges_features["second_position"] = positions_2[0]
edges_features = pd.concat([edges_features, pd.get_dummies(edges_features["second_position"],dtype='int')], axis=1, join='inner') # effectively adds actual currency to dummy variables/columns
edges_features.drop(["Payment Format", "second_position"], axis=1, inplace=True) # drop the axiliary columns
edges_features.head()

#### --- One-hot encoding: time ---

In [ ]:
# DONE convert timestamps --> higher number --> more recent 
edges_features["Timestamp"] = ((pd.to_datetime(edges_features['Timestamp']).astype(int) // 10**9) - 1661990000) // 10 # does not interpret time well... circular definition for months --> sinus calculations
edges_features

In [ ]:
# split for normalization purposes: only time & payments
df_requires_normalization, second_df = split_dataframe(edges_features)
df_requires_normalization = normalize(df_requires_normalization)
edges_features = pd.concat([df_requires_normalization,second_df], axis=1)

In [ ]:
edges_features

### --- Y - Edge Feature Matrix

In [ ]:
y = edges_features.to_numpy()
# print(y[0:10])

# GRAPHICAL - nx

In [ ]:
graph_full = create_graph(links, edges_amount)

In [ ]:
print(graph_full)

### --- Visualization ---

In [ ]:
# DONE Creating smaller graph for visualization:
limit = 150
small_graph = create_graph(links, edges_amount, limit=limit)

In [ ]:
pos = nx.random_layout(small_graph) # shell, circular, spectral, spring, random,
plt.figure(figsize=(25, 15))  # Increase figure size

nx.draw(
    small_graph,
    pos,
    node_size=300,  # Reduce node size for better visibility
    with_labels=True,
    font_size=7,
    font_weight='bold',
    node_color='lightblue',  # Specify node color
    edge_color='gray',  # Specify edge color
    width=1,  # Adjust edge width
    arrows=True,  # Show arrows for directed edges
    arrowstyle='->',  # Specify arrow style
    arrowsize=20,  # Adjust arrow size
)

edge_labels = nx.get_edge_attributes(small_graph, 'label')
nx.draw_networkx_edge_labels(
    small_graph,
    pos,
    edge_labels=edge_labels,
    label_pos=0.5,  # Adjust label position along edges
    font_size=7,  # Adjust font size
    font_color='green',  # Specify font color
)

if 'limit' in locals():
    plt.title(f'Graph Visualization of first {limit} transactions')  # Add title to the plot
else:
    plt.title(f'Graph Visualization of all transactions')  # Add title to the plot
plt.axis('off')  # Hide axis
plt.show()

#### --- Statistical elements as additional features for nodes ---

In [ ]:
# Dictionaries
degree_of_centrality = nx.degree_centrality(small_graph) # closeness_centrality, eigenvector_centrality, betweenness_centrality
betweenness_centrality = nx.betweenness_centrality(small_graph)

# TODO add statistics to a new DataFrame 

node_stat_features = pd.DataFrame()
node_stat_features['account'] = degree_of_centrality.keys()
node_stat_features['degree_of_centrality'] = degree_of_centrality.values()
node_stat_features['betweenness_centrality'] = betweenness_centrality.values()
node_stat_features

# ADJACENCY MATRIX - nx

### --- Loading full graph ---

In [ ]:
print(len(edges_features))
print(len(links)) # number of transactions

In [ ]:
print(len(unique_accounts))
print(graph_full.__len__()) # number of nodes in the graph

In [ ]:
adjacency_matrix = nx.adjacency_matrix(graph_full)
adjacency_matrix

In [ ]:
print("Number of nodes:", graph_full.number_of_nodes())
print("Shape of adjacency matrix:", adjacency_matrix.shape)

#### number in adjacency matrix does not match edges --> creating alternative matching

In [ ]:
accounts = unique_accounts.reset_index(drop=True)
accounts['ID'] = accounts.index
mapping_dict = dict(zip(accounts['Accounts'], accounts['ID']))
data['From'] = data['Account'].map(mapping_dict)
data['To'] = data['Account.1'].map(mapping_dict)
data = data.drop(['Account', 'Account.1', 'From Bank', 'To Bank'], axis=1)
data


In [ ]:
edge_index = torch.stack([torch.from_numpy(data['From'].values), torch.from_numpy(data['To'].values)], dim=0)
print(edge_index)
print(edge_index.size())

In [ ]:
adjacency_matrix = adjacency_matrix.todense()
adjacency_matrix

In [ ]:
print(adjacency_matrix) # index of nodes
print(len(adjacency_matrix))

In [ ]:
print(type(adjacency_matrix))

In [ ]:
adjacency_matrix = torch.from_numpy(adjacency_matrix).to(torch.float)
print(adjacency_matrix.size())

In [ ]:
num_ones = (adjacency_matrix == 1).sum().item()
print("Number of ones:", num_ones)

In [ ]:
# edge_index = torch.tensor(np.array(adjacency_matrix.nonzero()), dtype=torch.long)
# edge_index
# print(edge_index.size())

In [ ]:
node_features = node_features.to_numpy()
edges_features = edges_features.to_numpy()

In [ ]:
print(node_features)
print(edges_features)

In [ ]:
node_features = torch.from_numpy(node_features).to(torch.float)
edges_features = torch.from_numpy(edges_features).to(torch.float)
labels = torch.from_numpy(labels).to(torch.float)

In [ ]:
print(node_features.size())
print(edges_features.size())
print(edge_index.size())
print(labels.size())

In [ ]:
input_data = Data(
    x=node_features,
    edge_index=edge_index,
    edge_attr=edges_features,
    y=labels
)

print(input_data)
print(input_data.y.size())

In [ ]:
print(input_data)

In [ ]:
import pickle

with open("Saved-Data/graph.pickle", "wb") as f:
    pickle.dump({
        'edges_features': edges_features,
        'links': links,
        'unique_accounts': unique_accounts,
        'graph_full': graph_full,
        'adjacency_matrix': adjacency_matrix,
        'node_features': node_features,
        'edge_index': edge_index,
        'labels': labels,
        'input_data': input_data
    }, f)

# -------- !!! TODO !!! ----------

In [ ]:
# PLAN MONDAY 
# FIX Model input Data 
# TODO Create equivalent variables but flip nodes with edges 

# DONE Understand architecture 
# DONE Understand model input data 
# DONE There are missing connections in the adjacency matrix # IDEA scrap adjacency matrix entirely if necessary? 
# DONE Create model without NeighborSampler 
# DONE Clean code 
# DONE Make model work ... 

# PLAN TUESDAY 
# TODO CODE - Make model more extensive 
# TODO CODE - Visualize prediction patterns and learning curve for existing model 
# TODO CODE - Create testing stage and evaluation 
# TODO ADMIN - Send email with Update (ask about servers?
# TODO THEORY - Watch GNN Theory 
# TODO THEORY - Write up Introduction 

# PLAN WEDNESDAY 
# TODO THEORY - Understand Math Theory 
# TODO THEORY - Write up methodology theory 
# TODO THEORY - Write up Literature Review 

# PLAN THURSDAY 
# TODO CODE - add statistics to a new DataFrame 
# TODO CODE - add average of transactions for accounts 